# Biology & genomics primer for data scientists

This notebook is for **competitors without a biology or medicine background**. It gives a compact map from **DNA** to the **expression matrices** you will use in this challenge. No exam at the end — use it to decode terminology in papers and documentation.

**What this challenge actually gives you:** processed **gene × cell** count matrices in **AnnData** (`.h5ad`). You do *not* need to run alignment yourself to compete. The sections below explain *where those matrices come from* conceptually.


## 1. Human genome, genetic code, and “genes”

**Genome** means the full DNA sequence an organism inherits (for humans, two copies — one from each parent — of **23 pairs of chromosomes**). The haploid human genome is on the order of **3 billion** **nucleotide** letters.

**Alphabet.** DNA uses four letters: **A, T, G, C** (adenine, thymine, guanine, cytosine). **A** pairs with **T**, **G** with **C** on the two strands of the double helix.

**Genetic code (translation).** Proteins are chains of **amino acids**. The cell reads **mRNA** in non-overlapping triplets called **codons**. Most codons specify one of **20** amino acids; some signal **stop**. This mapping (codon → amino acid) is the **genetic code**. It applies when **ribosomes** translate **mRNA** into protein — not when we merely store DNA sequence.

**What is a “gene”, informally?** A **gene** is a named segment of DNA that can be transcribed into RNA and often (for protein-coding genes) translated into protein. In databases, a gene has an **identifier** (e.g. Ensembl **ENSG…** IDs) and **coordinates** on a **reference genome** build (e.g. **GRCh38 / hg38**).

**Annotation.** Researchers agree on a **reference assembly** and attach **annotations**: where exons, transcripts, and genes lie. Your `.h5ad` **variable** (`var`) names are typically **gene IDs** or symbols tied to that annotation. Expression is **not** “one number for the whole genome”; it is **one count (or normalized value) per gene (or transcript) per cell**.


## 2. Molecular flow: DNA → RNA → protein (central dogma)

**Replication:** DNA → DNA (when cells divide).

**Transcription:** DNA → **RNA**. The main product for protein-coding genes is **messenger RNA (mRNA)**. Eukaryotic genes often contain **introns** (spliced out) and **exons** (spliced together in mature mRNA).

**Translation:** mRNA → **protein** (amino-acid chain) on ribosomes, using the genetic code.

![Central dogma: DNA → RNA → protein](../images/Central-Dogma.png)

**Why RNA-seq?** It measures **abundance of RNA transcripts** (often mature mRNA), which reflects **which genes are active** in the sampled cells — not the DNA sequence itself and not protein abundance directly (though expression often correlates with protein levels).


## 3. Next-generation sequencing (NGS)

**NGS** refers to **high-throughput** methods that sequence **many millions of short DNA fragments** in parallel (Illumina is common). Typical outputs are **short reads** (tens to a few hundred bases).

**FASTQ** files store **reads** and **per-base quality scores** (how confident the sequencer is about each letter).

**Paired-end** sequencing reads both ends of each fragment; that helps map reads uniquely to repetitive regions of the genome.

Downstream of the instrument, bioinformatics turns FASTQ files into **alignments** (where each read maps on the reference) or **pseudoalignments** (k-mer matches without full base-by-base alignment), then into **counts per gene**.


## 4. RNA-seq versus DNA sequencing

| | **DNA-seq** (e.g. whole-genome / exome) | **Bulk RNA-seq** | **Single-cell RNA-seq** |
|---|----------------------------------------|------------------|-------------------------|
| **Measures** | DNA sequence (variants, structure) | Average RNA abundance across *many cells* in one sample | RNA abundance **per cell** |
| **Typical use** | Genotyping, somatic mutations, CNVs | Tissue-level expression | Cell types & states in a heterogeneous sample |

This challenge is built on **single-cell RNA-seq**–derived expression — not on raw DNA variant calls.


## 5. Multiplexing and single-cell RNA-seq

<video width="720" controls muted playsinline>
  <source src="../images/microfluidics.webm" type="video/webm"/>
</video>

*Droplet microfluidics: partitioning cells and reagents for barcoding (illustrative; vendor workflows vary).*

**Multiplexing (bulk).** Many **libraries** (samples) are pooled and sequenced in **one lane** or run. Each sample carries a **sample barcode** so reads can be sorted after sequencing.

**Single-cell RNA-seq** adds another layer: each **cell** must be labelled. Commercial droplet systems (e.g. **10x Chromium**) encapsulate single cells with barcoded beads so each read carries a **cell barcode**. **UMIs** (unique molecular identifiers) tag individual RNA molecules to reduce duplicate counting after PCR.

**Resulting data:** a **count matrix** — genes × cells (or cells × genes, depending on software) — where entry *(g, c)* is how many transcripts from gene *g* were observed in cell *c* (possibly after correction).


## 6. Typical bioinformatics pipeline (FASTQ → counts)

![From raw scRNA-seq reads to expression matrices (workflow overview)](../images/scrna-seq_data.png)

Rough stages (details vary by lab and protocol):

1. **QC** — filter low-quality reads, trim adapters; summarize quality (e.g. FastQC).
2. **Alignment or pseudoalignment** — map reads to a **reference transcriptome** or genome (tools such as STAR, HISAT2, or pseudoaligners like **kallisto** / **salmon**).
3. **Quantification** — aggregate reads to **genes** (or transcripts) → **count matrix**.
4. **Optional downstream** — normalization, batch correction, clustering, annotation.

**This repository skips steps 1–3 for you:** competition files are already **processed matrices** in `.h5ad` (and related tables). Understanding the pipeline helps you interpret **QC columns** in `obs` (e.g. mitochondrial percentage) and **gene identifiers** in `var`.


## 7. Containers for matrix data: RDS/Seurat, Loom, AnnData (`.h5ad`)

These are **not** raw sequencer output — they are **analysis objects** that bundle the count matrix with metadata:

| Container | Ecosystem | Notes |
|-----------|-----------|--------|
| **Seurat object** (often saved as **`.rds`** in R) | R / Bioconductor | Widely used in scRNA-seq tutorials; `Read10X`, `CreateSeuratObject`, etc. |
| **Loom** (`.loom`) | Python/R | HDF5-based; historically used with **loompy** / velocity workflows; less central today than h5ad in Python |
| **AnnData** (`.h5ad`) | Python (**Scanpy**, scverse) | Stores `X`, `obs`, `var`, `layers`, … — **this challenge uses `.h5ad`** |

You can convert between ecosystems (e.g. Seurat ↔ AnnData) with tools like `sceasy` or `anndata2ri`, but for the competition you only need **AnnData** concepts from notebook `01_anndata_and_pseudobulk`.


### Further reading (optional)

- NHGRI: [Introduction to genomics](https://www.genome.gov/about-genomics/fact-sheets/Introduction-to-Genomics)
- **10x Genomics** learning hub: [What is single cell RNA sequencing?](https://www.10xgenomics.com/what-is-single-cell-rna-seq)
- **Scanpy** / **AnnData** tutorials: [scverse](https://scverse.org/)

*Notebook `00` — conceptual background only; no code cells required to read along.*
